# Spread vs precision

Compute cached layer maps and rank spread/alignment metrics by F1 and by correlation with drop. No best/worst image examples are rendered here.

In [ ]:
from pathlib import Path
import sys

LAB_ROOT = Path.cwd()
if (LAB_ROOT / "classifier_patch_analysis").exists():
    REPO_ROOT = LAB_ROOT.parent
else:
    REPO_ROOT = LAB_ROOT
    LAB_ROOT = REPO_ROOT / "classifier_experiments"
sys.path.insert(0, str(LAB_ROOT))

In [ ]:
from classifier_patch_analysis import ClassifierAttackConfig, ClassifierPatchExperiment, ExperimentConfig

SEED = 17
IMG_SIZE = 224
MODEL_CHECKPOINT = str(REPO_ROOT / "data" / "yolo11-cls-person.pt")
BASE_WEIGHTS = str(REPO_ROOT / "data" / "yolo11s-cls.pt")
PATCH_PATH = str(REPO_ROOT / "data" / "cls_patch.png")
DATASET_PATH = str(REPO_ROOT / "datasets" / "COCO_people_224")
DEVICE = "auto"
BATCH_SIZE = 64
SUCCESS_DROP_THRESHOLD = 0.5
TARGET_LAYER = "model.9"

attack_config = ClassifierAttackConfig(
    dataset_path=DATASET_PATH,
    patch_path=PATCH_PATH,
    checkpoint_path=MODEL_CHECKPOINT,
    base_weights_path=BASE_WEIGHTS,
    output_dir=str(LAB_ROOT / "outputs" / "classifier_patch_analysis"),
    img_size=IMG_SIZE,
    device=DEVICE,
    success_drop_threshold=SUCCESS_DROP_THRESHOLD,
    inference_batch_size=BATCH_SIZE,
    seed=SEED,
)
exp = ClassifierPatchExperiment(ExperimentConfig(attack=attack_config, target_layer=TARGET_LAYER))
exp.output_dir, exp.figures_dir

In [ ]:
from classifier_patch_analysis import compute_or_load_layer_maps, compute_or_load_spread_vs_precision

MAX_EXAMPLES = None
FORCE_LAYER_MAPS = False
FORCE_TABLE = False

cache = exp.build_or_load_cache()
layer_map_status = compute_or_load_layer_maps(exp, layers=[TARGET_LAYER], max_examples=MAX_EXAMPLES, force=FORCE_LAYER_MAPS)
layer_map_status

In [ ]:
result = compute_or_load_spread_vs_precision(
    exp,
    layer_name=TARGET_LAYER,
    max_examples=MAX_EXAMPLES,
    top_percent=5.0,
    force=FORCE_TABLE,
)
rows_df = result["rows_df"]
quality = result["quality"]
regression = result["regression"]
display(quality.head(30))
display(regression.head(30))
rows_df.to_csv(exp.output_dir / "spread_vs_precision_rows.csv", index=False)
quality.to_csv(exp.output_dir / "spread_vs_precision_quality.csv", index=False)
regression.to_csv(exp.output_dir / "spread_vs_precision_drop_regression.csv", index=False)

In [ ]:
from classifier_patch_analysis.spread_precision import compute_layer_spread_summary
from classifier_patch_analysis.plots import (
    plot_quality_leaderboard,
    plot_regression_leaderboard,
    plot_layer_spread_map,
    plot_metric_vs_drop,
    savefig,
)

fig = plot_quality_leaderboard(quality, top_n=25)
savefig(fig, exp.figures_dir / "spread_precision_balanced_accuracy_leaderboard.png")
fig = plot_regression_leaderboard(regression, top_n=25)
savefig(fig, exp.figures_dir / "spread_precision_drop_regression.png")
summary = compute_layer_spread_summary(exp, layers=[TARGET_LAYER], max_examples=MAX_EXAMPLES)
display(summary)
fig = plot_layer_spread_map(summary)
savefig(fig, exp.figures_dir / "spread_precision_layer_spread_map.png")
if not quality.empty:
    fig = plot_metric_vs_drop(rows_df, quality.iloc[0]["metric"])
    savefig(fig, exp.figures_dir / "spread_precision_best_metric_vs_drop.png")